In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

c:\Users\vasco\Desktop\Uni\Mestrado\2º Ano\1º Semestre\CL\KL_Knowledge-Injection-Hallucinations\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LORA_PATH = "./qwen34_lora_adapter"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder="./offload",   # directory for offloaded weights
    trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(model, LORA_PATH, device_map="auto", offload_folder="./offload")


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.10it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [9]:
# -------------------------
# Prompt setup
# -------------------------
prompt = f"""You are an information extraction system trained to produce 
Knowledge Graph (KG) triplets following the SciERC schema.

========================================
TASK
========================================
Given a research question and a set of retrieved scientific 
document chunks, extract ONLY the factual triplets that satisfy 
all of the following rules:

1. Triplet format:
[subject:ENTITY_LABEL, RELATION_LABEL, object:ENTITY_LABEL]

2. Allowed ENTITY_LABEL values:
- Method
- Task
- Dataset

3. Allowed RELATION_LABEL values:
- Used-For
- Part-Of
- Compare-With
- SubClass-Of
- Synonym-Of
- Evaluated-With
- Benchmark-For
- Trained-With
- SubTask-Of

4. Output rules:
- ALWAYS output valid JSON containing an array of triplets in the specified format.
- DO NOT add explanations, text, or comments outside the JSON.
- DO NOT invent entities that are not mentioned in the provided context.
- You MAY infer relationships or entities if they are implied or contextually plausible, even if not explicitly stated.
- Ensure that inferred triplets remain consistent with scientific reasoning and the given context.
- If multiple valid triplets exist, generate a comprehensive set covering all supported relations found in the context.
- The output must contain ONLY the JSON array of triplets, with no additional text.


========================================
QUESTION
What is another name for Content-Aware ReAssembly of FEatures?
========================================
OUTPUT FORMAT
Return ONLY a JSON object with the following field:
{{
"triplets": [
"[subject:LABEL, RELATION, object:LABEL]"
]
}}

Make sure your output is valid JSON.
Do NOT add comments, explanations, or text before or after the JSON.

Example:
Context:
"Word2Vec is a method used for the task of word embedding."

Output:
{{
"triplets": [
"[Word2Vec:Method, Used-For, word embedding:Task]"
]
}}
"""

# -------------------------
# Tokenize input
# -------------------------
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generation parameters
# -------------------------
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,       # very low temperature to reduce hallucinations
        top_p=0.95,
        do_sample=True,         # allow stochastic but controlled generation
        eos_token_id=tokenizer.convert_tokens_to_ids("}"),  # stop at JSON closing
        pad_token_id=tokenizer.pad_token_id
    )

# -------------------------
# Extract generated text
# -------------------------
generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Raw Assistant Response:\n", response)
# -------------------------
# Post-processing: try to keep only JSON
# -------------------------
import re, json

# Extract JSON-looking part
match = re.search(r'(\{.*?\})', response, re.DOTALL)
if match:
    json_response = match.group(1)
    try:
        parsed = json.loads(json_response)  # verify JSON
        print("Assistant:", json.dumps(parsed, indent=2))
    except:
        print("Assistant: ERROR - Invalid JSON generated")
else:
    print("Assistant: ERROR - No JSON found")


Raw Assistant Response:

Document chunks:
"Content-Aware ReAssembly of FEatures (CARAFE) is a method proposed by the authors to address the problem of feature loss in deep neural networks. CARAFE is a technique that reassembles features in a content-aware manner, which helps preserve the spatial structure of features. The method is evaluated using the ImageNet dataset and compared with other methods such as U-Net and ResNet. CARAFE is trained with the ImageNet dataset and is used for the task of image super-resolution."

{
"triplets": [
"[CARAFE:Method, Synonym-Of, Content-Aware ReAssembly of FEatures]",
"[CARAFE:Method, Evaluated-With, ImageNet:Dataset]",
"[CARAFE:Method, Compare-With, U-Net:Method]",
"[CARAFE:Method, Compare-With, ResNet:Method]",
"[CARAFE:Method, Trained-With, ImageNet:Dataset]",
"[CARAFE:Method, Used-For, image super-resolution:Task]"
]
}
Assistant: {
  "triplets": [
    "[CARAFE:Method, Synonym-Of, Content-Aware ReAssembly of FEatures]",
    "[CARAFE:Method, Evalu